# Stage 8 — Vanilla STGCN on PEMS-BAY

Reproduces STGCN (Yu et al., IJCAI 2018) on plain PEMS-BAY (1 channel: speed).
Self-contained — pulls data and defines all code inline so it runs on any GPU kernel.

In [1]:
%pip install pyyaml scipy -q
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.11.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB


## Pull PEMS-BAY data

In [2]:
import os, subprocess, pathlib

DATA_DIR = pathlib.Path("pems_bay_data")
REPO_DIR = DATA_DIR / "Augmented-PEMS-BAY"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/george-j-ste/Augmented-PEMS-BAY.git",
         str(REPO_DIR)],
        check=True,
    )
    print("Cloned PEMS-BAY data.")
else:
    print("Data already present.")

SPEED_CSV       = REPO_DIR / "data" / "traffic_data" / "speed.csv"
SENSOR_IDS_TXT  = REPO_DIR / "data" / "sensor_graph" / "graph_sensor_ids.txt"
DISTANCES_CSV   = REPO_DIR / "data" / "sensor_graph" / "distances_bay_2017.csv"

assert SPEED_CSV.exists(), f"Missing {SPEED_CSV}"
print(f"Speed CSV: {SPEED_CSV} ({SPEED_CSV.stat().st_size / 1e6:.1f} MB)")

Cloned PEMS-BAY data.
Speed CSV: pems_bay_data/Augmented-PEMS-BAY/data/traffic_data/speed.csv (85.7 MB)


## Contract constants + data loader

In [8]:
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

# ── Contract constants ──
INPUT_WINDOW = 12
HORIZON = 12
SPLIT = {"train": 0.70, "val": 0.10, "test": 0.20}
EVAL_HORIZON_STEPS = {"15min": 3, "30min": 6, "60min": 12}

# ── Config ──
CFG = {
    "Kt": 3,
    "Ks": 3,
    "blocks": [[2, 64, 64], [64, 64, 64]],
    "dropout": 0.3,
    "lr": 0.001,
    "weight_decay": 0.0001,
    "batch_size": 64,
    "epochs": 100,
    "patience": 20,
    "n_seeds": 1,
    "base_seed": 42,
}


# ── Data loading ──
def load_adjacency(sensor_ids_path, distances_path, normalized_k=0.1):
    with open(sensor_ids_path) as f:
        sensor_ids = f.read().strip().split(",")
    dist_df = pd.read_csv(distances_path, dtype={"from": str, "to": str})
    n = len(sensor_ids)
    id_to_idx = {sid: i for i, sid in enumerate(sensor_ids)}
    dist_mx = np.full((n, n), np.inf, dtype=np.float32)
    for row in dist_df.itertuples(index=False):
        fr, to = str(row[0]), str(row[1])
        if fr in id_to_idx and to in id_to_idx:
            dist_mx[id_to_idx[fr], id_to_idx[to]] = row[2]
    std = dist_mx[~np.isinf(dist_mx)].std()
    adj = np.exp(-np.square(dist_mx / std))
    adj[adj < normalized_k] = 0
    return adj, sensor_ids


def load_speed(speed_path, sensor_ids):
    """Load speed.csv -> [T, N] speed array + [T] time-of-day array."""
    df = pd.read_csv(speed_path, index_col=0)
    int_ids = [int(sid) for sid in sensor_ids]
    df = df.loc[int_ids]
    df = df.interpolate(axis=1).ffill(axis=1).bfill(axis=1)

    timestamps = pd.to_datetime(df.columns)
    seconds_in_day = (
        timestamps.hour * 3600 + timestamps.minute * 60 + timestamps.second
    )
    time_of_day = (seconds_in_day / 86400).values.astype(np.float32)

    return df.values.T.astype(np.float32), time_of_day


def make_samples(speed, time_of_day, input_window, horizon):
    """Sliding-window samples. Returns X [n, T_in, N, 2], Y [n, T_out, N]."""
    T, N = speed.shape
    n = T - input_window - horizon + 1
    X = np.empty((n, input_window, N, 2), dtype=np.float32)
    Y = np.empty((n, horizon, N), dtype=np.float32)
    for i in range(n):
        X[i, :, :, 0] = speed[i : i + input_window]
        X[i, :, :, 1] = time_of_day[i : i + input_window, np.newaxis]
        Y[i] = speed[i + input_window : i + input_window + horizon]
    return X, Y


def build_loaders(speed_path, sensor_ids_path, distances_path, batch_size):
    adj, sensor_ids = load_adjacency(sensor_ids_path, distances_path)
    speed, time_of_day = load_speed(speed_path, sensor_ids)
    X, Y = make_samples(speed, time_of_day, INPUT_WINDOW, HORIZON)
    n = len(X)
    n_train = int(n * SPLIT["train"])
    n_val = int(n * SPLIT["val"])

    # Z-score speed channel only (channel 0), matching reference.
    # Time-of-day (channel 1) stays raw [0, 1].
    # Y is NOT normalized — loss/metrics operate on raw mph.
    mean = float(X[:n_train, :, :, 0].mean())
    std = float(X[:n_train, :, :, 0].std())
    X[:, :, :, 0] = (X[:, :, :, 0] - mean) / std

    splits = {
        "train": (X[:n_train], Y[:n_train]),
        "val": (X[n_train:n_train + n_val], Y[n_train:n_train + n_val]),
        "test": (X[n_train + n_val:], Y[n_train + n_val:]),
    }
    loaders = {}
    for name, (x, y) in splits.items():
        ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
        loaders[name] = DataLoader(ds, batch_size=batch_size, shuffle=(name == "train"))

    scaler = {"mean": mean, "std": std}
    return loaders, adj, scaler


print("Data loader ready.")

Data loader ready.


## STGCN model

In [9]:
import torch.nn as nn
from scipy.sparse.linalg import eigsh


def scaled_laplacian(adj):
    n = adj.shape[0]
    d = adj.sum(axis=1)
    d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)
    lap = np.eye(n) - np.diag(d_inv_sqrt) @ adj @ np.diag(d_inv_sqrt)
    lambda_max = eigsh(lap, k=1, which="LM", return_eigenvectors=False)[0]
    return (2.0 / lambda_max) * lap - np.eye(n)


def cheb_polynomials(scaled_lap, K):
    n = scaled_lap.shape[0]
    polys = [np.eye(n, dtype=np.float32)]
    if K > 1:
        polys.append(scaled_lap.astype(np.float32))
    for _ in range(2, K):
        polys.append((2.0 * scaled_lap @ polys[-1] - polys[-2]).astype(np.float32))
    return [torch.from_numpy(p) for p in polys]


class ChebConv(nn.Module):
    def __init__(self, K, c_in, c_out):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(K, c_in, c_out))
        self.bias = nn.Parameter(torch.zeros(c_out))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, cheb_polys):
        B, T, N, _ = x.shape
        x_flat = x.reshape(B * T, N, -1)
        out = torch.zeros(B * T, N, self.weight.shape[2], device=x.device, dtype=x.dtype)
        for k, poly in enumerate(cheb_polys):
            transformed = torch.matmul(poly.to(x.device), x_flat)
            out = out + torch.matmul(transformed, self.weight[k])
        return (out + self.bias).reshape(B, T, N, -1)


class TemporalConv(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv2d(c_in, 2 * c_out, kernel_size=(kernel_size, 1))

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        out = self.conv(x)
        p, q = out.chunk(2, dim=1)
        return (p * torch.sigmoid(q)).permute(0, 2, 3, 1)


class STConvBlock(nn.Module):
    def __init__(self, Ks, Kt, c_in, c_mid, c_out, dropout=0.0):
        super().__init__()
        self.temp1 = TemporalConv(c_in, c_mid, Kt)
        self.graph = ChebConv(Ks, c_mid, c_mid)
        self.temp2 = TemporalConv(c_mid, c_out, Kt)
        self.norm = nn.LayerNorm(c_out)
        self.dropout = nn.Dropout(dropout)
        self.residual_conv = nn.Conv2d(c_in, c_out, kernel_size=1) if c_in != c_out else None
        self.Kt = Kt

    def forward(self, x, cheb_polys):
        residual = x
        out = self.temp1(x)
        out = self.graph(out, cheb_polys)
        out = self.temp2(out)
        out = self.norm(out)
        out = self.dropout(out)
        trim = self.Kt - 1
        if trim > 0:
            residual = residual[:, trim:-trim, :, :]
        if self.residual_conv is not None:
            residual = self.residual_conv(residual.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        return torch.relu(out + residual)


class STGCN(nn.Module):
    def __init__(self, cheb_polys, c_in, input_window, horizon, Ks, Kt, blocks, dropout):
        super().__init__()
        self.n_cheb = len(cheb_polys)
        for k, poly in enumerate(cheb_polys):
            self.register_buffer(f"_cheb_{k}", poly)
        self.st_blocks = nn.ModuleList()
        for channels in blocks:
            self.st_blocks.append(STConvBlock(Ks, Kt, channels[0], channels[1], channels[2], dropout))
        t_remaining = input_window - len(blocks) * 2 * (Kt - 1)
        assert t_remaining > 0
        c_last = blocks[-1][-1]
        self.output_conv = nn.Conv2d(c_last, c_last, kernel_size=(t_remaining, 1))
        self.fc = nn.Linear(c_last, horizon)

    @property
    def _cheb_polys(self):
        return [getattr(self, f"_cheb_{k}") for k in range(self.n_cheb)]

    def forward(self, x):
        polys = self._cheb_polys
        for block in self.st_blocks:
            x = block(x, polys)
        x = x.permute(0, 3, 1, 2)
        x = self.output_conv(x).squeeze(2)
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        return x.permute(0, 2, 1)  # [B, horizon, N]


print(f"Model defined.")

Model defined.


## Load data and build model

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

print("Loading data …")
loaders, adj, scaler = build_loaders(
    str(SPEED_CSV), str(SENSOR_IDS_TXT), str(DISTANCES_CSV), CFG["batch_size"]
)
print(f"Scaler: mean={scaler['mean']:.2f} mph, std={scaler['std']:.2f}")
for name, loader in loaders.items():
    print(f"  {name}: {len(loader.dataset)} samples")

# Build graph
L = scaled_laplacian(adj)
cheb_polys = cheb_polynomials(L, CFG["Ks"])


def build_model():
    return STGCN(
        cheb_polys=cheb_polys, c_in=2,
        input_window=INPUT_WINDOW, horizon=HORIZON,
        Ks=CFG["Ks"], Kt=CFG["Kt"],
        blocks=CFG["blocks"], dropout=CFG["dropout"],
    ).to(device)


model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

# Quick forward pass check
x, y = next(iter(loaders["train"]))
x = x.to(device)
pred = model(x)
print(f"Input: {x.shape} → Output: {pred.shape}")

Device: cuda
Loading data …
Scaler: mean=62.74 mph, std=9.44
  train: 36465 samples
  val: 5209 samples
  test: 10419 samples
Parameters: 117,388
Input: torch.Size([64, 12, 325, 2]) → Output: torch.Size([64, 12, 325])


## Training utilities

In [11]:
import time


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _build_mask(true, null_val=0.0):
    """Normalized mask that excludes null_val entries (matching Graph WaveNet)."""
    mask = (true != null_val).float()
    mask = mask / torch.mean(mask)
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    return mask


def _masked_mean(raw_loss, mask):
    loss = raw_loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)


def masked_mae_loss(pred, true, null_val=0.0):
    """Masked MAE used as training loss."""
    mask = _build_mask(true, null_val)
    return _masked_mean(torch.abs(pred - true), mask)


def compute_metrics(pred, true, null_val=0.0):
    """MAE, RMSE, MAPE on de-normalized values, masking null_val entries."""
    mask = _build_mask(true, null_val)
    diff = pred - true
    abs_diff = torch.abs(diff)

    mae = _masked_mean(abs_diff, mask).item()
    rmse = torch.sqrt(_masked_mean(diff ** 2, mask)).item()
    mape = _masked_mean(abs_diff / torch.abs(true), mask).item() * 100

    return {"mae": mae, "rmse": rmse, "mape": mape}


def train_epoch(model, loader, optimizer):
    model.train()
    mean, std = scaler["mean"], scaler["std"]
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        pred_mph = pred * std + mean
        loss = masked_mae_loss(pred_mph, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        n += 1
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        preds.append(model(x))
        trues.append(y)
    pred = torch.cat(preds) * scaler["std"] + scaler["mean"]
    true = torch.cat(trues)  # already raw mph

    horizon_metrics = []
    for i in range(pred.shape[1]):
        horizon_metrics.append(compute_metrics(pred[:, i, :], true[:, i, :]))

    results = {
        "all": {
            "mae": float(np.mean([m["mae"] for m in horizon_metrics])),
            "rmse": float(np.mean([m["rmse"] for m in horizon_metrics])),
            "mape": float(np.mean([m["mape"] for m in horizon_metrics])),
        }
    }
    for name, step in EVAL_HORIZON_STEPS.items():
        results[name] = horizon_metrics[step - 1]
    return results


print("Training utilities ready.")

Training utilities ready.


## Train (3 seeds)

In [12]:
os.makedirs("checkpoints", exist_ok=True)

seeds = [CFG["base_seed"] + i for i in range(CFG["n_seeds"])]
all_results = []

for seed in seeds:
    print(f"\n{'=' * 60}")
    print(f"Seed {seed}")
    print(f"{'=' * 60}")
    set_seed(seed)

    model = build_model()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"],
    )

    best_val_mae = float("inf")
    patience_counter = 0
    ckpt_path = f"checkpoints/stgcn_seed{seed}.pt"

    for epoch in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        train_loss = train_epoch(model, loaders["train"], optimizer)
        val_results = evaluate(model, loaders["val"])

        val_mae = val_results["all"]["mae"]
        elapsed = time.time() - t0

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | Train Loss {train_loss:.4f} | "
                  f"Val MAE {val_mae:.4f} mph | {elapsed:.1f}s")

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= CFG["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

    # Test with best checkpoint
    model.load_state_dict(
        torch.load(ckpt_path, weights_only=True, map_location=device)
    )
    test_results = evaluate(model, loaders["test"])
    all_results.append(test_results)

    print(f"\nTest results (seed {seed}):")
    for horizon, metrics in test_results.items():
        print(f"  {horizon:>5s}: MAE={metrics['mae']:.4f}  "
              f"RMSE={metrics['rmse']:.4f}  MAPE={metrics['mape']:.2f}%")


Seed 42
Epoch   1 | Train Loss 2.2706 | Val MAE 2.2676 mph | 14.9s
Epoch  10 | Train Loss 1.8246 | Val MAE 1.9741 mph | 15.1s
Epoch  20 | Train Loss 1.7833 | Val MAE 1.9442 mph | 14.9s
Epoch  30 | Train Loss 1.7486 | Val MAE 1.9072 mph | 15.0s
Epoch  40 | Train Loss 1.7263 | Val MAE 1.8521 mph | 14.9s
Epoch  50 | Train Loss 1.7073 | Val MAE 1.8622 mph | 15.0s
Epoch  60 | Train Loss 1.6958 | Val MAE 1.8376 mph | 15.0s
Epoch  70 | Train Loss 1.6849 | Val MAE 1.8075 mph | 14.9s
Epoch  80 | Train Loss 1.6779 | Val MAE 1.8074 mph | 14.9s
Early stopping at epoch 81

Test results (seed 42):
    all: MAE=1.7149  RMSE=3.7196  MAPE=3.84%
  15min: MAE=1.3834  RMSE=2.8685  MAPE=2.90%
  30min: MAE=1.7663  RMSE=3.8887  MAPE=3.96%
  60min: MAE=2.1807  RMSE=4.8528  MAPE=5.11%


## Summary

Published PEMS-BAY targets: MAE ≈ 1.36, RMSE ≈ 2.96, MAPE ≈ 2.9% (60 min).

In [13]:
rows = []
for horizon in all_results[0]:
    maes  = [r[horizon]["mae"]  for r in all_results]
    rmses = [r[horizon]["rmse"] for r in all_results]
    mapes = [r[horizon]["mape"] for r in all_results]
    rows.append({
        "Horizon": horizon,
        "MAE":  f"{np.mean(maes):.4f} ± {np.std(maes):.4f}",
        "RMSE": f"{np.mean(rmses):.4f} ± {np.std(rmses):.4f}",
        "MAPE": f"{np.mean(mapes):.2f} ± {np.std(mapes):.2f}%",
    })

summary_df = pd.DataFrame(rows).set_index("Horizon")
summary_df

,MAE,RMSE,MAPE
Horizon,,,
all,1.7149 ± 0.0000,3.7196 ± 0.0000,3.84 ± 0.00%
15min,1.3834 ± 0.0000,2.8685 ± 0.0000,2.90 ± 0.00%
30min,1.7663 ± 0.0000,3.8887 ± 0.0000,3.96 ± 0.00%
60min,2.1807 ± 0.0000,4.8528 ± 0.0000,5.11 ± 0.00%
